# V19 – Aufgaben: Musterlösungen

Vollständige Lösungen zu den fünf Aufgaben aus [03_aufgaben.ipynb](../03_aufgaben.ipynb). Schau erst rein, nachdem du es selbst versucht hast.


## Aufgabe 1 – Tabelle erstellen


In [1]:
import sqlite3

def erstelle_tabelle(cursor):
    cursor.execute('''CREATE TABLE Produkte (
        Produkt_ID INTEGER PRIMARY KEY,
        Produktname TEXT NOT NULL,
        Produktionszeit_Minuten INTEGER,
        Material_pro_Stueck_kg REAL
    )''')

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
erstelle_tabelle(cur)
print(cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())


[('Produkte',)]


## Aufgabe 2 – Einfügen mit Parameter-Binding


In [2]:
def fuege_produkt_ein(cursor, produkt_id, name, zeit, material):
    cursor.execute(
        "INSERT INTO Produkte VALUES (?, ?, ?, ?)",
        (produkt_id, name, zeit, material),
    )

fuege_produkt_ein(cur, 1, "Zahnrad Z42", 25, 1.8)
fuege_produkt_ein(cur, 2, "Welle W-18", 35, 3.2)
print(cur.execute("SELECT * FROM Produkte").fetchall())


[(1, 'Zahnrad Z42', 25, 1.8), (2, 'Welle W-18', 35, 3.2)]


## Aufgabe 3 – Alle Produkte lesen


In [3]:
def lese_alle_produkte(cursor):
    return cursor.execute(
        "SELECT * FROM Produkte ORDER BY Produkt_ID"
    ).fetchall()

print(lese_alle_produkte(cur))


[(1, 'Zahnrad Z42', 25, 1.8), (2, 'Welle W-18', 35, 3.2)]


## Aufgabe 4 – CSV importieren und filtern


In [4]:
import csv

def importiere_produkte(cursor, csv_pfad):
    with open(csv_pfad, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for zeile in reader:
            cursor.execute(
                "INSERT INTO Produkte VALUES (?, ?, ?, ?)",
                (
                    int(zeile["Produkt_ID"]),
                    zeile["Produktname"],
                    int(zeile["Produktionszeit_Minuten"]),
                    float(zeile["Material_pro_Stueck_kg"]),
                ),
            )

def produkte_ueber_zeit(cursor, min_zeit):
    rows = cursor.execute(
        "SELECT Produktname FROM Produkte WHERE Produktionszeit_Minuten > ? ORDER BY Produktname",
        (min_zeit,),
    ).fetchall()
    return [row[0] for row in rows]

# Frische DB
conn4 = sqlite3.connect(":memory:")
cur4 = conn4.cursor()
erstelle_tabelle(cur4)
importiere_produkte(cur4, "../testdaten/produkte.csv")
print("Alle:", lese_alle_produkte(cur4))
print("Ueber 20 min:", produkte_ueber_zeit(cur4, 20))


Alle: [(1, 'Zahnrad Z42', 25, 1.8), (2, 'Welle W-18', 35, 3.2), (3, 'Flansch F-90', 18, 2.5), (4, 'Buchse B-25', 12, 0.8), (5, 'Gehaeuse G-55', 45, 5.5), (6, 'Verbindungselement VE-12', 8, 0.3)]
Ueber 20 min: ['Gehaeuse G-55', 'Welle W-18', 'Zahnrad Z42']


## Aufgabe 5 – UPDATE und DELETE


In [5]:
def erhoehe_zeit(cursor, produkt_id, delta):
    cursor.execute(
        "UPDATE Produkte SET Produktionszeit_Minuten = Produktionszeit_Minuten + ? WHERE Produkt_ID = ?",
        (delta, produkt_id),
    )

def loesche_produkt(cursor, produkt_id):
    cursor.execute(
        "DELETE FROM Produkte WHERE Produkt_ID = ?",
        (produkt_id,),
    )

vor = cur4.execute("SELECT Produktionszeit_Minuten FROM Produkte WHERE Produkt_ID = 3").fetchone()[0]
erhoehe_zeit(cur4, 3, 5)
nach = cur4.execute("SELECT Produktionszeit_Minuten FROM Produkte WHERE Produkt_ID = 3").fetchone()[0]
print(f"Produkt 3: {vor} -> {nach} Minuten")

loesche_produkt(cur4, 6)
print("Zeilen nach DELETE:", cur4.execute("SELECT COUNT(*) FROM Produkte").fetchone()[0])


Produkt 3: 18 -> 23 Minuten
Zeilen nach DELETE: 5


## ✅ Fertig
Alle fünf Aufgaben sind sauber gelöst. Zurück zur [README](../README.md).
